<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/report_7_QM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPORT: Quantum chemistry with ORCA

YOUR NAME

In [ ]:
from datetime import date

today = date.today()
print(today)

Fill in the **training** part (everybody) and then the part of **your group**: GROUP 1 (noncovalent interactions) or GROUP 2 (colour of biological chromophores). Inside each group every member studies a different system (see the tables below; if you are more than five, two members can take the same system with different methods, agree on this with the teacher). The final section of each group, the comparison of all the systems, is written together.

Include the figures that are asked (you can make them in the notebook or with VMD/ChimeraX, and add them with `Insert → Image`) and keep the code cells you used, so that your results can be reproduced.

## Setup

Run these cells at the beginning of each session: they install ORCA from your Google Drive (see the training notebook) and define the same helper functions.

In [ ]:
import os, glob, shutil, subprocess

ORCA_DIR = os.environ.get('ORCA_DIR', '')      # on your own computer: the folder containing the program orca
DRIVE_FOLDER = '/content/drive/MyDrive/Structural_Bioinformatics'

if not (ORCA_DIR and os.path.isfile(os.path.join(ORCA_DIR, 'orca'))):
    from google.colab import drive
    drive.mount('/content/drive')
    archives = sorted(glob.glob(os.path.join(DRIVE_FOLDER, 'ORCA', 'orca_6*linux_x86-64*.tar.*')))
    if not archives:
        raise FileNotFoundError('No ORCA archive found in ' + os.path.join(DRIVE_FOLDER, 'ORCA') +
                                ': follow the instructions above to download it.')
    archive = archives[-1]                        # if there is more than one, take the most recent version
    ORCA_DIR = os.path.join('/content', os.path.basename(archive).split('.tar')[0])
    if not os.path.isfile(os.path.join(ORCA_DIR, 'orca')):
        print('unpacking', os.path.basename(archive), '...')
        subprocess.run(['tar', '-xf', archive, '-C', '/content'], check=True)
    # some archives contain an additional folder level
    found = glob.glob(os.path.join('/content', 'orca_6*', '**', 'orca'), recursive=True)
    ORCA_DIR = os.path.dirname(sorted(found)[-1])

# ORCA must be called with its full path; the shared version also needs its libraries
os.environ['PATH'] = ORCA_DIR + ':' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = ORCA_DIR + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMP_NUM_THREADS'] = '1'
ORCA = os.path.join(ORCA_DIR, 'orca')
version = subprocess.run([ORCA, '--version'], capture_output=True, text=True).stdout.strip()
print('ORCA found in', ORCA_DIR)
print(version if version else 'ORCA is installed')

In [ ]:
%pip install -q py3Dmol

In [ ]:
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()
if os.path.isdir('/content/drive/MyDrive'):          # on Colab, with Google Drive mounted
    local_path = '/content/T07'
    DRIVE_PATH = os.path.join(DRIVE_FOLDER, 'T07')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    shutil.copytree(DRIVE_PATH, local_path, dirs_exist_ok=True)    # bring back the results of previous sessions
else:
    local_path = os.path.join(BASE_DIR, 'T07')
    DRIVE_PATH = None
os.makedirs(local_path, exist_ok=True)
%cd $local_path
print('results are saved in', DRIVE_PATH if DRIVE_PATH else local_path)

# the input structures are in the Data/qm folder of the course repository:
# if this notebook is run from a copy of the repository we use it, otherwise we download the repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'qm', 's22')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = '/content/sb' if os.path.isdir('/content') else os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
QM_DIR = os.path.join(REPO_DIR, 'Data', 'qm')
print('input files from', QM_DIR)

In [ ]:
import re, time
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol

HARTREE2EV = 27.211386        # 1 Eh in eV
HARTREE2KCAL = 627.5095       # 1 Eh in kcal/mol
BOHR2ANG = 0.529177           # 1 bohr in Å

def save_to_drive(workdir):
    # copy the useful files of a calculation (not the large temporary ones) to the same folder on Google Drive
    if not DRIVE_PATH:
        return
    target = os.path.join(DRIVE_PATH, os.path.relpath(os.path.abspath(workdir), local_path))
    os.makedirs(target, exist_ok=True)
    for f in os.listdir(workdir):
        if f.endswith(('.inp', '.out', '.xyz', '.cube', '.csv', '.hess')):
            shutil.copy2(os.path.join(workdir, f), target)

def run_orca(name, inp, folder='.', rerun=False):
    # write folder/name/name.inp, run ORCA in folder/name and return the path of the output file
    workdir = os.path.join(folder, name)
    os.makedirs(workdir, exist_ok=True)
    out = os.path.join(workdir, name + '.out')
    inpfile = os.path.join(workdir, name + '.inp')
    inp = inp.strip() + '\n'
    if (not rerun and os.path.isfile(out) and 'ORCA TERMINATED NORMALLY' in open(out).read()
            and os.path.isfile(inpfile) and open(inpfile).read() == inp):
        print(f'{name}: already done, reading {out}')
        return out
    for f in os.listdir(workdir):            # a new run starts from an empty folder
        if os.path.isfile(os.path.join(workdir, f)):
            os.remove(os.path.join(workdir, f))
    with open(inpfile, 'w') as f:
        f.write(inp)
    t0 = time.time()
    with open(out, 'w') as f:
        subprocess.run([ORCA, name + '.inp'], cwd=workdir, stdout=f, stderr=subprocess.STDOUT)
    text = open(out).read()
    if 'ORCA TERMINATED NORMALLY' not in text:
        print(text[-3000:])
        raise RuntimeError(f'ORCA did not terminate normally: look at the end of {out}')
    print(f'{name}: done in {time.time() - t0:.1f} s')
    save_to_drive(workdir)
    return out

def xyz_block(xyzfile, ghost=()):
    # the coordinate lines of an xyz file; the atoms whose index is in ghost become ghost atoms (see Part 4)
    lines = open(xyzfile).read().splitlines()
    natoms = int(lines[0])
    atoms = []
    for i, l in enumerate(lines[2:2 + natoms]):
        el, x, y, z = l.split()[:4]
        atoms.append(f'{el}{":" if i in ghost else " "} {x:>12} {y:>12} {z:>12}')
    return '\n'.join(atoms)

def final_energy(out):
    # the last 'FINAL SINGLE POINT ENERGY' of an output (in Eh)
    return float(re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)', open(out).read())[-1])

def wall_time(out):
    # total run time of a calculation, in seconds
    d, h, m, s, ms = re.findall(r'TOTAL RUN TIME:\s+(\d+) days (\d+) hours (\d+) minutes (\d+) seconds (\d+) msec', open(out).read())[-1]
    return 86400*int(d) + 3600*int(h) + 60*int(m) + int(s) + int(ms)/1000

def s2(out):
    # expectation value of S^2 (only for UHF/UKS calculations)
    return float(re.findall(r'Expectation value of <S\*\*2>\s*:\s*(-?\d+\.\d+)', open(out).read())[-1])

def dipole(out):
    # magnitude of the dipole moment, in Debye
    return float(re.findall(r'Magnitude \(Debye\)\s*:\s*(\d+\.\d+)', open(out).read())[-1])

def orbital_energies(out):
    # the last table of orbital energies: list of (occupation, energy in eV); for UHF the spin-up orbitals
    text = open(out).read().split('ORBITAL ENERGIES')[-1]
    rows = []
    for l in text.splitlines()[4:]:
        f = l.split()
        if len(f) != 4:
            if rows: break
            continue
        rows.append((float(f[1]), float(f[3])))
    return rows

def charges(out, kind='MULLIKEN'):
    # atomic charges: kind is MULLIKEN, LOEWDIN, HIRSHFELD (needs the keyword Hirshfeld) or CHELPG (keyword CHELPG)
    text = open(out).read()
    if kind == 'MULLIKEN':
        block = text.split('MULLIKEN ATOMIC CHARGES')[-1].split('Sum of atomic charges')[0]
    elif kind == 'LOEWDIN':
        block = text.split('LOEWDIN ATOMIC CHARGES')[-1].split('LOEWDIN REDUCED')[0]
    elif kind == 'HIRSHFELD':
        block = text.split('HIRSHFELD ANALYSIS')[-1].split('TOTAL')[0]
    elif kind == 'CHELPG':
        block = text.split('CHELPG Charges')[-1].split('Total charge')[0]
    return [(el, float(q)) for el, q in re.findall(r'^\s*\d+\s+([A-Z][a-z]?)\s*:?\s+(-?\d+\.\d+)', block, re.M)]

def read_xyz(xyzfile):
    lines = open(xyzfile).read().splitlines()
    n = int(lines[0])
    el = [l.split()[0] for l in lines[2:2+n]]
    xyz = np.array([[float(v) for v in l.split()[1:4]] for l in lines[2:2+n]])
    return el, xyz

def read_cube(cubefile):
    # Gaussian cube file -> (origin, axes, values); origin and axes in bohr
    with open(cubefile) as f:
        f.readline(); f.readline()
        n, *origin = f.readline().split()
        natoms = abs(int(n))
        axes, shape = [], []
        for _ in range(3):
            npt, *v = f.readline().split()
            shape.append(int(npt)); axes.append([float(x) for x in v])
        for _ in range(natoms):
            f.readline()
        data = np.array(f.read().split(), dtype=float)
    return np.array(origin, dtype=float), np.array(axes), data[-np.prod(shape):].reshape(shape)

def show_molecule(xyzfile, width=400, height=300):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()
    return view.show()

def cube_for_display(cubefile, maxpoints=40):
    # the text of a cube file with at most maxpoints per side and 4 digits: much smaller in the notebook
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    origin, axes, data = read_cube(cubefile)
    step = [max(1, int(np.ceil(n / maxpoints))) for n in data.shape]
    data = data[::step[0], ::step[1], ::step[2]]
    head = lines[:2] + [lines[2]]
    for i in range(3):
        v = axes[i] * step[i]
        head.append(f'{data.shape[i]:5d} {v[0]:12.6f} {v[1]:12.6f} {v[2]:12.6f}\n')
    head += lines[6:6 + natoms]
    values = data.ravel()
    body = '\n'.join(' '.join(f'{x:.4g}' for x in values[k:k + 6]) for k in range(0, len(values), 6))
    return ''.join(head) + body + '\n'

def cube_atoms_xyz(cubefile):
    # the atoms stored in a cube file (in bohr) as an xyz text (in Å)
    elements = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 16: 'S'}
    with open(cubefile) as f:
        lines = f.readlines()
    natoms = abs(int(lines[2].split()[0]))
    atoms = []
    for l in lines[6:6 + natoms]:
        z, _, x, y, w = l.split()[:5]
        atoms.append(f'{elements.get(int(float(z)), "X")} {float(x) * BOHR2ANG:.4f} {float(y) * BOHR2ANG:.4f} {float(w) * BOHR2ANG:.4f}')
    return f'{natoms}\n\n' + '\n'.join(atoms) + '\n'

def show_cube(cubefile, isovalue=0.05, signed=False, width=400, height=300):
    # isosurface of a cube file; signed=True for orbitals (positive lobes red, negative lobes blue)
    view = py3Dmol.view(width=width, height=height)
    view.addModel(cube_atoms_xyz(cubefile), 'xyz')
    view.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.2}})
    data = cube_for_display(cubefile)
    view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8})
    if signed:
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8})
    view.zoomTo()
    return view.show()

def plots_block(cubes, npoints=50):
    # an ORCA %plots block that writes cube files; cubes is a list of strings like
    # 'ElDens("x.eldens.cube")' or 'MO("x.mo4.cube",4,0)' (orbitals are numbered from 0)
    return '%plots\n  dim1 {0}\n  dim2 {0}\n  dim3 {0}\n  Format Gaussian_Cube\n'.format(npoints) + \
           ''.join(f'  {c};\n' for c in cubes) + 'end\n'

In [ ]:
def interaction_energy(xyzfile, nA, method, folder, counterpoise=True, q=(0, 0, 0)):
    # interaction energy (kcal/mol) of a dimer whose first nA atoms are molecule A and the others molecule B
    # q = (charge of AB, charge of A, charge of B); all species are closed-shell singlets
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))   # no spaces, / or () in file names
    A, B = range(nA), range(nA, nat)
    jobs = {'AB': (xyz_block(xyzfile), q[0])}
    if counterpoise:                                     # monomers in the basis of the dimer: the partner is made of ghost atoms
        jobs['A'] = (xyz_block(xyzfile, ghost=B), q[1])
        jobs['B'] = (xyz_block(xyzfile, ghost=A), q[2])
    else:                                                # monomers in their own basis: the partner is removed
        lines = xyz_block(xyzfile).splitlines()
        jobs['A'] = ('\n'.join(lines[:nA]), q[1])
        jobs['B'] = ('\n'.join(lines[nA:]), q[2])
    E = {}
    for k, (coords, charge) in jobs.items():
        name = f'{tag}_{k}' + ('_cp' if counterpoise and k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{coords}\n*', folder=folder)
        E[k] = final_energy(out)
    return (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL

In [ ]:
def excitations(out):
    # excited states from the absorption spectrum table: list of (energy eV, wavelength nm, oscillator strength)
    text = open(out).read().split('ABSORPTION SPECTRUM VIA TRANSITION ELECTRIC DIPOLE MOMENTS')[1].split('\n\n')[0]
    rows = []
    for l in text.splitlines():
        if '->' in l:
            f = l.split('->')[1].split()
            rows.append((float(f[1]), float(f[3]), float(f[4])))
    return np.array(rows)

def transitions(out):
    # the orbital contributions to each excited state, as printed by ORCA (e.g. "7a -> 8a : 0.98")
    text = open(out).read().split('EXCITED STATES')[1].split('ABSORPTION SPECTRUM')[0]
    for block in re.split(r'\n(?=STATE\s+\d+:)', text)[1:]:
        head = block.splitlines()[0]
        contr = re.findall(r'(\d+a\s+->\s+\d+a)\s+:\s+(\d\.\d+)', block)
        print(head.split('<S**2>')[0].strip(), '  ', ',  '.join(f'{t} ({c})' for t, c in contr))

def spectrum(states, fwhm=0.3, emin=1.5, emax=8.0):
    # absorption spectrum: a Gaussian of width fwhm (eV) for each state, with area proportional to f
    E = np.linspace(emin, emax, 1000)
    sigma = fwhm / 2.3548
    eps = sum(f * np.exp(-(E - e)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) for e, lam, f in states)
    return E, eps

In [ ]:
def dispersion(out):
    # the D3/D4 dispersion correction (Eh) included in the final energy; 0 if there is none
    v = re.findall(r'Dispersion correction\s+(-?\d+\.\d+)', open(out).read())
    return float(v[-1]) if v else 0.0

def interaction_energy_split(xyzfile, nA, method, folder, q=(0, 0, 0)):
    # counterpoise-corrected interaction energy (kcal/mol) split in the electronic part and the dispersion correction:
    # with a method like 'B3LYP D4 def2-TZVP' one calculation gives both B3LYP and B3LYP-D4
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + re.sub(r'[^A-Za-z0-9_+-]', '', method.replace(' ', '_'))   # no spaces, / or () in file names
    A, B = range(nA), range(nA, nat)
    E, D = {}, {}
    for k, ghost, charge in [('AB', (), q[0]), ('A', B, q[1]), ('B', A, q[2])]:
        name = f'{tag}_{k}' + ('_cp' if k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{xyz_block(xyzfile, ghost=ghost)}\n*', folder=folder)
        E[k], D[k] = final_energy(out), dispersion(out)
    total = (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL
    disp = (D['AB'] - D['A'] - D['B']) * HARTREE2KCAL
    return total - disp, disp            # (without dispersion, dispersion contribution)

def optimize(name, xyzfile, charge=0, method='Native-GFN2-XTB', extra='', folder='.'):
    # geometry optimisation; returns the xyz file of the optimised structure
    out = run_orca(name, f'! {method} Opt TightSCF {extra}\n* xyz {charge} 1\n{xyz_block(xyzfile)}\n*', folder=folder)
    return os.path.join(os.path.dirname(out), name + '.xyz')

## Training (everybody)

Answer briefly (a few lines each), with the numbers and figures from the training notebook. You can copy the outputs, or recompute them here: the calculations are already in your `T07` folder and are just read again.

### Part 1: the hydrogen atom

In [ ]:
for method in ['HF def2-SVP', 'HF def2-TZVP', 'HF def2-QZVPP', 'r2SCAN-3c']:
    name = 'H_' + method.replace(' ', '_')
    out = run_orca(name, f'! {method} TightSCF\n* xyz 0 2\nH 0 0 0\n*', folder='H')
    print(f'{method:16s} {final_energy(out):10.6f} Eh   error {(final_energy(out) + 0.5) * HARTREE2KCAL:6.2f} kcal/mol')

*   **Table**: the energies of the hydrogen atom and their errors with respect to the exact value (−0.5 E$_h$). Why does the HF error become (almost) zero with the largest basis set, while the DFT error does not?
*   **Figure**: the radial distribution $4\pi r^2\rho(r)$, computed vs exact. Where is its maximum? Where does the calculation deviate from the exact result, and why?

**Answer:**

### Part 2: the carbon atom

*   Which spin state is the most stable? What is its multiplicity, and how many unpaired electrons does it have?
*   **Figure**: the spin density of the triplet. Where are the unpaired electrons? Why isn't the spin density spherical? (Think about the shape of the $p$ orbitals.)

**Answer:**

### Part 3: the chemical bond in H$_2$

*   **Figure**: the bonding and antibonding orbitals, and the density difference $\Delta\rho$. Where do the electrons accumulate when the bond forms?
*   **Figure**: the energy curves. Compare bond length and bond energy of the three methods with experiment (0.741 Å, 4.75 eV). Why does RHF fail when the atoms are far apart?
*   How large is the correlation energy (CCSD − RHF) at the minimum, in kcal/mol? Compare it with the energy of a hydrogen bond (≈ 5 kcal/mol): can we neglect it?

**Answer:**

### Part 4: water

*   **Table**: geometry, bending frequency, dipole and computer time with HF/def2-SVP and r$^2$SCAN-3c, and the experimental values. Which method is better? How much more expensive is it?
*   **Figure**: the HOMO of water. Where is it located? How is it related to the ability of water to accept hydrogen bonds?
*   Compare the CHELPG charges with those of TIP3P. The charges of a water model used in MD describe water in the liquid, where each molecule is polarised by its neighbours: is it reasonable that they are larger?
*   Water dimer: **table** of the interaction energy with and without the counterpoise correction. What is the BSSE, and how does it depend on the basis set?

**Answer:**

### Part 5: dynamics

*   **Figure**: the position of the excess proton vs time, for two simulations with different seeds. How long does the proton take to reach the ammonia? Why are the two simulations different?
*   Look at the movie: do the oxygen atoms move a lot while the proton travels? Explain in your words why the charge moves faster than the atoms (the Grotthuss mechanism).
*   Why can't this simulation be done with a classical force field such as the one used in T04?

**Answer:**

### Part 6: excited states

*   Formaldehyde: what are the HOMO and the LUMO (**figure**)? Why is the first excitation dark ($f \approx 0$)?
*   **Figure**: the spectra of formaldehyde and butadiene. Is any of the two molecules coloured? What would you expect for a molecule with 5 or 10 conjugated double bonds?

**Answer:**

## GROUP 1: noncovalent interactions in biomolecules

The structure of proteins and nucleic acids, the recognition between a ligand and its receptor, the double helix: all are held together by **noncovalent interactions**, each of a few kcal/mol, comparable with $k_BT$ (0.6 kcal/mol at 300 K). They have different physical origins:

*   **electrostatics**, the interaction between the permanent charge distributions of the molecules (in a hydrogen bond, the positive H of a donor and the lone pair of an acceptor);
*   **induction** (polarisation), the deformation of the electron density of one molecule by the field of the other;
*   **dispersion** (London forces), the correlation between the instantaneous fluctuations of the electrons of the two molecules: it is always attractive and dominates the stacking of aromatic rings;
*   **exchange repulsion** (Pauli), when the electron clouds overlap.

HF contains electrostatics, induction and exchange, but no correlation, so **no dispersion**. Many density functionals (B3LYP among them) don't describe dispersion either, and need an empirical correction such as D3 or D4: $E_{disp} = -\sum_{AB} f(R_{AB})\, C_6^{AB}/R_{AB}^6 - \dots$, the same $1/r^6$ of the Lennard-Jones potential. In a force field, electrostatics is described by fixed point charges and dispersion + exchange by the Lennard-Jones term; induction is missing (or included on average in the charges).

We use the complexes of the **S22 database** (Jurečka et al., *Phys. Chem. Chem. Phys.* 8, 1985 (2006)), with CCSD(T)/CBS reference interaction energies (Takatani et al., *J. Chem. Phys.* 132, 144104 (2010)), and the **S22x5** extension (Gráfová et al., *J. Chem. Theory Comput.* 6, 2365 (2010)), where each complex is also given with the intermolecular distance scaled by 0.9, 1.2, 1.5 and 2.0, with reference values. All files are in `Data/qm/s22`: in the xyz files the atoms of molecule A come first (`nA` atoms), then those of molecule B.

**Each member chooses one pair of complexes**: the same molecules (or similar ones) in two different arrangements.

| Member | Complex 1 | Complex 2 | Biological relevance |
|:--:|:--|:--|:--|
| 1 | adenine–thymine Watson–Crick (7) | adenine–thymine stacked (15) | base pairing vs base stacking in DNA |
| 2 | uracil dimer, H-bonded (5) | uracil dimer, stacked (13) | base pairing vs stacking in RNA |
| 3 | indole–benzene stacked (14) | indole–benzene T-shaped (21) | Trp–Phe contacts in protein cores |
| 4 | formamide dimer (4) | formic acid dimer (3) | backbone N–H···O=C bonds, carboxylic acids (Asp, Glu) |
| 5 | phenol dimer (22) | benzene–water (17) | Tyr–Tyr contacts, O–H···π interactions |

Write here your choice (S22 numbers, e.g. `7` and `15`):

In [ ]:
my_complexes = [7, 15]          # the S22 numbers of your two complexes

import pandas as pd
s22 = pd.read_csv(os.path.join(QM_DIR, 's22', 's22x5_reference.csv'))
mine = s22[s22.id.isin(my_complexes)]
mine

In [ ]:
def s22_file(i, scale=1.0):
    row = s22[(s22.id == i) & (s22.scale == scale)].iloc[0]
    return os.path.join(QM_DIR, 's22', row.file), int(row.nA), float(row.Eint_ref_kcal)

for i in my_complexes:
    show_molecule(s22_file(i)[0])

### 1. Interaction energies at the equilibrium geometry

Compute the counterpoise-corrected interaction energy of your two complexes with HF, B3LYP and B3LYP-D4 (def2-SVP basis set) and with r$^2$SCAN-3c. A calculation with dispersion gives two numbers: `interaction_energy_split` separates the contribution of the D4 correction, so one B3LYP-D4 calculation gives both B3LYP and B3LYP-D4 (and r$^2$SCAN-3c gives r$^2$SCAN with and without D4).

**Timings** (on Colab, one core): for the largest complexes (adenine–thymine, 30 atoms) a single calculation takes about 1 minute with HF/def2-SVP and r$^2$SCAN-3c, 2 minutes with B3LYP-D4/def2-SVP and 7 minutes with def2-TZVP; each interaction energy needs 3 calculations. The smaller complexes are much faster. Work on Google Drive (as set up above): if Colab disconnects, the calculations already done are not lost and are not repeated when you run the cell again.

In [ ]:
os.makedirs('group1', exist_ok=True)
rows = []
for i in my_complexes:
    xyz, nA, ref = s22_file(i)
    hf, _ = interaction_energy_split(xyz, nA, 'HF def2-SVP', 'group1')
    b3, d4 = interaction_energy_split(xyz, nA, 'B3LYP D4 def2-SVP', 'group1')
    r2, r2d = interaction_energy_split(xyz, nA, 'r2SCAN-3c', 'group1')
    rows.append({'complex': i, 'HF/SVP': hf, 'B3LYP/SVP': b3, 'B3LYP-D4/SVP': b3 + d4, 'D4 (B3LYP)': d4,
                 'r2SCAN-3c no D4': r2, 'r2SCAN-3c': r2 + r2d, 'reference': ref})
table1 = pd.DataFrame(rows).set_index('complex').round(2)
table1

*   **Table**: interaction energies and errors with respect to the reference, for each method. Which methods are acceptable for your complexes (errors below 1 kcal/mol)?
*   Which fraction of the interaction energy comes from the dispersion correction in the two complexes? Relate it to the nature of the interaction.

**Answer:**

### 2. The basis set and the basis-set superposition error

For **one** of your two complexes (the H-bonded one, if you have one), compute the B3LYP-D4 interaction energy with def2-SVP and def2-TZVP, with and without counterpoise correction (`interaction_energy` from the training). The calculations with def2-SVP and the counterpoise-corrected ones are already done in the previous step and are just read again; the def2-TZVP ones take longer (about 25 minutes for adenine–thymine).

In [ ]:
i = my_complexes[0]
xyz, nA, ref = s22_file(i)
for basis in ['def2-SVP', 'def2-TZVP']:
    e_nocp = interaction_energy(xyz, nA, f'B3LYP D4 {basis}', 'group1', counterpoise=False)
    e_cp = sum(interaction_energy_split(xyz, nA, f'B3LYP D4 {basis}', 'group1'))
    print(f'complex {i:2d} {basis:10s}  no CP: {e_nocp:7.2f}   CP: {e_cp:7.2f}   BSSE = E(CP) - E(no CP): {e_cp - e_nocp:5.2f}   reference: {ref:7.2f} kcal/mol')

*   How large is the BSSE, in absolute value and as a fraction of the interaction energy, with the two basis sets? Is the uncorrected def2-SVP result closer to or further from the reference than the corrected one? (Sometimes errors cancel: a good result for the wrong reason.)
*   r$^2$SCAN-3c uses a small basis set too, but it includes an empirical correction for the BSSE (gCP): compare.

**Answer:**

### 3. The potential energy curve

Now compute the interaction energy along the S22x5 curve (distances scaled by 0.9, 1.0, 1.2, 1.5, 2.0) with HF/def2-SVP and r$^2$SCAN-3c (with and without its D4 term), and compare with the CCSD(T) reference. For adenine–thymine this takes about 30 minutes per complex: start it, and in the meantime work on the questions above.

In [ ]:
scales = [0.9, 1.0, 1.2, 1.5, 2.0]
fig, axs = plt.subplots(1, len(my_complexes), figsize=(5.5 * len(my_complexes), 4), squeeze=False)
curves1 = {}
for ax, i in zip(axs[0], my_complexes):
    hf, r2, r2d, ref = [], [], [], []
    for s in scales:
        xyz, nA, r = s22_file(i, s)
        hf.append(interaction_energy_split(xyz, nA, 'HF def2-SVP', 'group1')[0])
        e, d = interaction_energy_split(xyz, nA, 'r2SCAN-3c', 'group1')
        r2.append(e); r2d.append(e + d); ref.append(r)
    curves1[i] = pd.DataFrame({'scale': scales, 'HF/SVP': hf, 'r2SCAN-3c no D4': r2, 'r2SCAN-3c': r2d, 'reference': ref})
    for col, style in [('reference', 'ko-'), ('HF/SVP', 's--'), ('r2SCAN-3c no D4', '^--'), ('r2SCAN-3c', 'o-')]:
        ax.plot(scales, curves1[i][col], style, label=col)
    ax.axhline(0, c='grey', lw=0.5)
    ax.set_title(s22[s22.id == i].system.iloc[0]); ax.set_xlabel('distance / equilibrium distance'); ax.set_ylabel('interaction energy (kcal/mol)')
    ax.legend()
plt.tight_layout(); plt.show()
curves1

*   **Figure**: the curves of your two complexes. Which methods find the correct minimum? Are the stacked complexes bound at all without dispersion?
*   At which distance does the dispersion term become negligible? Compare its decay with the $1/r^6$ law (plot the D4 contribution vs the scale factor on a log-log scale).

**Answer:**

### 4. Electrostatics: where are the charges?

Compute the CHELPG charges and the dipole of the two monomers of one complex (in the geometry of the complex) and look at the atoms involved in the contact. In a force field, the electrostatic interaction is the Coulomb energy between point charges like these: the cell computes it ($332.06\, \sum_{i \in A} \sum_{j \in B} q_i q_j / r_{ij}$ kcal/mol, with $r$ in Å) along the S22x5 curve, keeping the charges fixed.

In [ ]:
i = my_complexes[0]
xyz, nA, ref = s22_file(i)
lines = xyz_block(xyz).splitlines()
q = {}
for k, coords in [('A', lines[:nA]), ('B', lines[nA:])]:
    out = run_orca(f's22_{i}_{k}_chelpg', '! r2SCAN-3c CHELPG TightSCF\n* xyz 0 1\n' + '\n'.join(coords) + '\n*', folder='group1')
    q[k] = np.array([c for _, c in charges(out, 'CHELPG')])
    print(f'molecule {k}: dipole {dipole(out):.2f} D')
    print('  '.join(f'{el}{n} {c:+.2f}' for n, (el, c) in enumerate(charges(out, 'CHELPG'), start=0 if k == 'A' else nA)))

def coulomb(xyzfile, nA, qA, qB):
    el, x = read_xyz(xyzfile)
    r = np.linalg.norm(x[:nA, None, :] - x[None, nA:, :], axis=2)
    return 332.06 * np.sum(np.outer(qA, qB) / r)

for s in [0.9, 1.0, 1.2, 1.5, 2.0]:
    f, _, r = s22_file(i, s)
    print(f'scale {s:3.1f}: Coulomb energy of the CHELPG charges {coulomb(f, nA, q["A"], q["B"]):7.2f}   reference {r:7.2f} kcal/mol')

*   Which atoms carry the largest charges? Are the contacts in the H-bonded complex between charges of opposite sign? And in the stacked one?
*   How does the Coulomb energy of the point charges compare with the reference along the curve, for the H-bonded and the stacked complex? What is missing in a model made only of point charges?

**Answer:**

### 5. Optional: a "gold standard" calculation

The reference values are CCSD(T)/CBS: CCSD plus a perturbative estimate of triple excitations, extrapolated to an infinite basis set. Canonical CCSD(T) scales as $N^7$ with the size of the system and is out of reach here, but its *local* approximation DLPNO-CCSD(T) is available in ORCA. With `interaction_energy(xyz, nA, 'DLPNO-CCSD(T) def2-TZVP def2-TZVP/C', 'group1')` you can compute it for a small complex: the water dimer takes less than a minute, the formamide dimer (12 atoms) about 15 minutes, the complexes with 30 atoms more than an hour. How close is it to the reference? What is still missing? (Hint: compare with the HF and B3LYP results obtained with the same basis set, and think about the basis set.)

**Answer:**

### Group discussion

Collect in a single **table** the interaction energies (reference, HF, B3LYP, B3LYP-D4, r$^2$SCAN-3c) of all the complexes studied by the group, ordered by strength, and classify them as hydrogen-bonded, stacked (dispersion) or mixed.

*   For the nucleobases: is base pairing or base stacking stronger in the gas phase? What would change in water, where the bases can make hydrogen bonds with the solvent instead? What does this suggest about the stability of the double helix?
*   Rank the methods. Why is HF reasonable for some complexes and not for others?
*   Which of these interactions can a classical force field (fixed point charges + Lennard-Jones) reproduce well, and which are more difficult? (Think about the directionality of the hydrogen bond, and about polarisation.)

**Answer:**

## GROUP 2: the colour of biological chromophores

Very few biomolecules absorb visible light: proteins and nucleic acids are colourless. Light is captured by **chromophores**, molecules with extended conjugated $\pi$ systems, often bound to a protein that tunes their absorption: retinal in rhodopsin (vision), the chromophore of the green fluorescent protein (GFP), flavins (redox enzymes, light sensors), chlorophylls. The absorption of a chromophore depends on the length of the conjugated system, on its protonation state and on its environment (solvent, protein).

We compute the vertical excitation energies with TD-DFT, as in Part 6 of the training. Each calculation is done in two steps: (1) optimisation of the ground-state geometry with GFN2-xTB, a fast semiempirical (tight-binding) method that takes seconds even for retinal, starting from the structures in `Data/qm/chromophores` (built from SMILES with RDKit); (2) TD-DFT at the optimised geometry, in the gas phase and in water described as a dielectric continuum (CPCM).

**Each member chooses one system:**

| Member | Molecules (files in `Data/qm/chromophores`) | Question | Experimental $\lambda_{max}$ (approximate) |
|:--:|:--|:--|:--|
| 1 | GFP chromophore HBDI: `hbdi_neutral`, `hbdi_anion` (phenolate), `hbdi_cation` | How does the protonation state change the colour? | water: neutral ≈ 370 nm, anion ≈ 425 nm, cation ≈ 395 nm; anion in vacuo 479 nm; in GFP: 395 nm (neutral) and 475 nm (anion) |
| 2 | retinal protonated Schiff base: `retinal_psb_alltrans`, `retinal_psb_11cis` | Why is retinal coloured and how does the protein tune it? | in vacuo ≈ 610 nm; in methanol ≈ 440 nm; rhodopsin (11-*cis*) 498 nm; bacteriorhodopsin (all-*trans*) 568 nm |
| 3 | protonated Schiff base polyenes: `psb_2` … `psb_6` (2 to 6 conjugated double bonds) | How does the absorption depend on the length of the conjugated chain? | (compare with the retinal values: retinal has 6 conjugated double bonds) |
| 4 | lumiflavin: `lumiflavin_ox` (oxidised), `lumiflavin_red` (reduced, 1,5-dihydro) | Why are oxidised flavins yellow and reduced ones colourless? | oxidised, water: ≈ 445 and ≈ 370 nm; reduced: no band in the visible |
| 5 | aromatic side chains: `toluene` (Phe), `p_cresol` (Tyr), `p_cresolate` (deprotonated Tyr), `3_methylindole` (Trp) | Why do proteins absorb at 280 nm? | Phe ≈ 257 nm, Tyr ≈ 275 nm, tyrosinate ≈ 293 nm, Trp ≈ 280 nm (weak bands); strong bands below 230 nm |

The experimental values are approximate: look for the original papers and cite them in the report.

**Timings** (Colab, one core, CAM-B3LYP/def2-SVP, 10 states): about 1 minute for the aromatic side chains, 5 minutes for HBDI (28 atoms) and lumiflavin, 15 minutes for retinal (54 atoms). Members 2 and 4 should start early, and can do the comparison of functionals (section 3) for one molecule only.

Write here your molecules:

In [ ]:
my_molecules = ['hbdi_neutral', 'hbdi_anion', 'hbdi_cation']

CHROMO_DIR = os.path.join(QM_DIR, 'chromophores')
def charge_of(name):
    # the charge is written in the second line of the xyz files
    return int(re.search(r'charge=([+-]?\d+)', open(os.path.join(CHROMO_DIR, name + '.xyz')).readlines()[1]).group(1))

for m in my_molecules:
    print(m, 'charge', charge_of(m))
    show_molecule(os.path.join(CHROMO_DIR, m + '.xyz'))

### 1. Ground-state geometries

Optimise your molecules with GFN2-xTB in the gas phase. Look at the optimised structures: are they planar? Check the bond lengths along the conjugated chain: is there a clear alternation of single and double bonds?

*Optional*: optimise your smallest molecule also with r$^2$SCAN-3c (`optimize(name, xyzfile, charge, method='r2SCAN-3c', folder='group2')`, 10–30 minutes) and compute its spectrum at the two geometries: how sensitive is the excitation energy to the geometry?

In [ ]:
os.makedirs('group2', exist_ok=True)
opt_xyz = {}
for m in my_molecules:
    opt_xyz[m] = optimize(m + '_opt', os.path.join(CHROMO_DIR, m + '.xyz'), charge=charge_of(m), folder='group2')
    show_molecule(opt_xyz[m])

### 2. Excited states in the gas phase

TD-DFT with CAM-B3LYP/def2-SVP, 10 excited states. For anions, diffuse functions can matter (their electrons are less bound): if you have an anion, repeat its calculation with `method='CAM-B3LYP def2-SVPD'` (about twice the time) and compare. The cell also writes the HOMO and the LUMO in cube files. The number of the HOMO is (number of electrons)/2 − 1, since ORCA numbers the orbitals from 0.

In [ ]:
def n_electrons(xyzfile, charge):
    Z = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'S': 16}
    el, _ = read_xyz(xyzfile)
    return sum(Z[e] for e in el) - charge

def tddft(name, xyzfile, charge, method='CAM-B3LYP def2-SVP', nroots=10, solvent=None, cubes=True):
    homo = n_electrons(xyzfile, charge) // 2 - 1
    keywords = f'! {method} TightSCF' + (f' CPCM({solvent})' if solvent else '')
    plots = plots_block([f'MO("{name}.homo.cube",{homo},0)', f'MO("{name}.lumo.cube",{homo + 1},0)']) if cubes else ''
    inp = f'{keywords}\n%tddft\n  NRoots {nroots}\nend\n{plots}* xyz {charge} 1\n{xyz_block(xyzfile)}\n*'
    return run_orca(name, inp, folder='group2')

def lowest_bright(states, fmin=0.01):
    # the lowest excited state with oscillator strength above fmin: the first absorption band we compare with experiment
    return states[np.argmax(states[:, 2] > fmin)]

tddft_out = {}
for m in my_molecules:
    tddft_out[m] = tddft(m + '_cam_gas', opt_xyz[m], charge_of(m))
    print(m)
    transitions(tddft_out[m])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for m in my_molecules:
    st = excitations(tddft_out[m])
    E, eps = spectrum(st, emin=1.5, emax=9.0)
    ax.plot(E, eps, label=m)
    ax.vlines(st[:, 0], 0, st[:, 2] * 3, colors=ax.lines[-1].get_color())
    for label, s in [('first state', st[0]), ('lowest bright (f > 0.01)', lowest_bright(st)), ('brightest', st[np.argmax(st[:, 2])])]:
        print(f'{m:22s} {label:25s} {s[0]:5.2f} eV  {s[1]:5.0f} nm  f = {s[2]:.3f}')
ax.axvspan(1239.84 / 700, 1239.84 / 400, color='yellow', alpha=0.15, label='visible')
ax.set_xlabel('excitation energy (eV)'); ax.set_ylabel('absorption (arb. units)'); ax.legend()
secax = ax.secondary_xaxis('top', functions=(lambda e: 1239.84 / np.maximum(e, 1e-3), lambda l: 1239.84 / np.maximum(l, 1e-3)))
secax.set_xlabel('wavelength (nm)')
plt.tight_layout(); plt.show()

In [ ]:
m = my_molecules[0]          # look at the orbitals of all your molecules
show_cube(f'group2/{m}_cam_gas/{m}_cam_gas.homo.cube', isovalue=0.03, signed=True)
show_cube(f'group2/{m}_cam_gas/{m}_cam_gas.lumo.cube', isovalue=0.03, signed=True)

*   **Table**: for each molecule, energy (eV), wavelength (nm) and oscillator strength of the first excited state, of the lowest bright state (the first absorption band, to compare with the experimental $\lambda_{max}$ in the table above) and of the brightest state, and the orbitals involved. Ten states cover only the lowest part of the spectrum: is that enough for your molecules? (You can increase `nroots`.)
*   **Figure**: HOMO and LUMO of your molecules. Is the bright state a $\pi \rightarrow \pi^*$ transition? Is there a charge transfer from one part of the molecule to another (compare where HOMO and LUMO are)?

**Answer:**

### 3. The effect of the functional

TD-DFT results depend on the functional, especially for long conjugated systems and for charge-transfer states. Repeat the calculation with B3LYP (20% exact exchange), PBE0 (25%) and ωB97X-D3 (range separated, like CAM-B3LYP), and compare the wavelength of the lowest bright state.

In [ ]:
func_molecules = my_molecules        # members 2 and 4: one molecule is enough, e.g. my_molecules[:1]
func_table = []
for m in func_molecules:
    row = {'molecule': m}
    for func in ['PBE0', 'B3LYP', 'CAM-B3LYP', 'wB97X-D3']:
        out = tddft_out[m] if func == 'CAM-B3LYP' else tddft(f'{m}_{func}_gas', opt_xyz[m], charge_of(m), method=f'{func} def2-SVP', cubes=False)
        st = excitations(out)
        row[func + ' (nm)'] = round(lowest_bright(st)[1])
    func_table.append(row)
pd.DataFrame(func_table).set_index('molecule')

*   How large is the spread of the results (in nm and in eV)? Is the ordering of your molecules the same with all functionals? Which functional is closest to experiment?

**Answer:**

### 4. The effect of the solvent

Repeat the CAM-B3LYP calculation in water with the CPCM continuum model (for consistency, one should also reoptimise the geometry in solvent, with `optimize(..., extra='CPCM(water)')`: try it for one molecule and see whether it matters).

In [ ]:
for m in my_molecules:
    out = tddft(m + '_cam_water', opt_xyz[m], charge_of(m), solvent='water', cubes=False)
    gas, wat = excitations(tddft_out[m]), excitations(out)
    bg, bw = lowest_bright(gas), lowest_bright(wat)
    print(f'{m:22s} gas: {bg[1]:5.0f} nm (f={bg[2]:.2f})   water: {bw[1]:5.0f} nm (f={bw[2]:.2f})   shift: {bw[0] - bg[0]:+.2f} eV')

*   Does the solvent shift the absorption to longer (red shift) or shorter (blue shift) wavelengths? Is the shift the same for all your molecules (e.g. charged vs neutral)?
*   Compare gas phase, water and experiment. A continuum model does not describe specific interactions such as hydrogen bonds with the chromophore: could this matter for your molecules?

**Answer:**

### 5. Your question

Answer the question of your system (see the table above), using your results:

*   **Member 1 (GFP)**: rationalise the shift between the neutral and the anionic form with the orbitals. In GFP the neutral form absorbs at 395 nm but emits green light (≈ 508 nm) from the anionic form: what has to happen in the excited state? (Look up *excited-state proton transfer*.)
*   **Member 2 (retinal)**: compare all-*trans* and 11-*cis*. The *opsin shift* is the difference between the absorption in the protein and in methanol: how large is it in eV for rhodopsin and bacteriorhodopsin? What in the protein could cause it (counterion, polarity of the pocket, twisting of the chain)? A simple test: put a negative point charge (a counterion) near the Schiff base nitrogen, with the `%pointcharges` keyword of ORCA, and see how the excitation changes.
*   **Member 3 (polyenes)**: plot the excitation energy vs the number of double bonds $N$ and vs $1/N$. In the *particle-in-a-box* model the $\pi$ electrons move freely along a chain of length $L \propto N$, and the HOMO→LUMO energy is $\Delta E = \frac{h^2}{8 m_e L^2}(N_\pi + 1)$ with $N_\pi = 2N$ electrons: how does $\Delta E$ scale with $N$? Does TD-DFT follow it? Extrapolate to $N = 6$ and compare with retinal.
*   **Member 4 (flavin)**: compare the orbitals involved in the bright transitions of the two forms. Why does the reduction (addition of two H atoms) remove the visible absorption? Which enzymes use flavins, and why is the colour change useful in the lab?
*   **Member 5 (aromatic side chains)**: which side chain absorbs most strongly above 250 nm? Why are the lowest bands of benzene-like molecules so weak? Why is the absorbance at 280 nm used to measure the protein concentration, and why does it depend on the sequence? What happens to Tyr at high pH?

**Answer:**

### Group discussion

*   **Figure**: all the computed (CAM-B3LYP, water) vs experimental $\lambda_{max}$ of the group, with the diagonal. Is the error systematic?
*   Which molecular features move the absorption into the visible (conjugation length, charge, protonation, charge transfer)?
*   Proteins tune the colour of their chromophores: summarise the mechanisms that came out in your systems. How would you model the chromophore *inside* the protein? (Think about QM/MM, lecture 8.)

**Answer:**